# PA3 Model / Runtime Prep (ONLINE, CPU only; separate source release)
Use two fresh notebooks for runtime and model. Internet ON, accelerator None.
Use the independently reviewed Task 6 PREP commit and pa3_prep_source.bundle. This source grants no inference rights. OFFLINE source stays at 13ea2fba67f36883518db8a2add1b249c951e48e.
See notes/pa3_task6_runtime_prep.md. Never substitute this bundle in the inference notebook.
The final cell output lists the exact files to download with their SHA-256 and size.


In [ ]:
KIND = "runtime"  # separate fresh notebook: "model"
PREP_SOURCE_COMMIT = ""  # exact Task 6 commit independently reviewed by ChatGPT/Antigravity
# Path relative to /kaggle/input exactly as Kaggle shows it, e.g.
# "datasets/<owner>/pa3-reviewed-prep-source/pa3_prep_source.bundle"
PREP_SOURCE_BUNDLE_INPUT = ""
PREP_SOURCE_BUNDLE_SHA256 = ""  # independently audited SHA-256 of that bundle
OFFLINE_SOURCE_COMMIT = "13ea2fba67f36883518db8a2add1b249c951e48e"  # fixed; do not edit


In [ ]:
import hashlib
import json
import os
from pathlib import Path, PurePosixPath
import re
import subprocess
import sys

work = Path.cwd()
inputs = work.parent / "input"
if KIND not in {"runtime", "model"} or not re.fullmatch(r"[0-9a-f]{40}", PREP_SOURCE_COMMIT):
    raise ValueError("SET_KIND_AND_REVIEWED_SOURCE_SHA")
if OFFLINE_SOURCE_COMMIT != "13ea2fba67f36883518db8a2add1b249c951e48e":
    raise ValueError("OFFLINE_SOURCE_COMMIT_IS_FIXED")
p = PurePosixPath(PREP_SOURCE_BUNDLE_INPUT)
if not PREP_SOURCE_BUNDLE_INPUT or p.is_absolute() or ".." in p.parts or "\\" in PREP_SOURCE_BUNDLE_INPUT or ":" in PREP_SOURCE_BUNDLE_INPUT:
    raise ValueError("INPUT_RELATIVE_SOURCE_BUNDLE_REQUIRED")
bundle = inputs / p
if not bundle.resolve().is_relative_to(inputs.resolve()):
    raise ValueError("BUNDLE_ESCAPES_INPUT")
def sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()
if sha256(bundle) != PREP_SOURCE_BUNDLE_SHA256:
    raise ValueError("SOURCE_BUNDLE_CHECKSUM")
receipt = bundle.parent / "pa3_prep_source_release.json"
if not receipt.is_file():
    raise ValueError("PREP_RELEASE_RECEIPT_MISSING_NEXT_TO_BUNDLE")
release = json.loads(receipt.read_text())
if (bundle.name != "pa3_prep_source.bundle"
    or release.get("schema") != "pa3-online-prep-source-v1"
    or release.get("prep_source_commit") != PREP_SOURCE_COMMIT
    or release.get("bundle_sha256") != PREP_SOURCE_BUNDLE_SHA256
    or release.get("offline_source_commit") != OFFLINE_SOURCE_COMMIT
    or release.get("execution_authority") is not False):
    raise ValueError("SEPARATE_REVIEWED_PREP_RELEASE_REQUIRED")
repo = work / "data/processed" / ("pa3-prep-" + KIND)
if repo.exists():
    raise FileExistsError("FRESH_PREP_NOTEBOOK_REQUIRED")
subprocess.run(["git", "-c", "core.autocrlf=false", "clone", "--no-checkout", str(bundle), str(repo)], check=True)
subprocess.run(["git", "-C", str(repo), "-c", "core.autocrlf=false", "checkout", "--detach", PREP_SOURCE_COMMIT], check=True)
actual = subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True).strip()
if actual != PREP_SOURCE_COMMIT:
    raise ValueError("PREP_CHECKOUT_MISMATCH")
env = {**os.environ, "CUDA_VISIBLE_DEVICES": "", "PYTHONPATH": str(repo), "PYTHONNOUSERSITE": "1"}
env.pop("PYTHONHOME", None)
if KIND == "model":
    subprocess.run([sys.executable, "-m", "pip", "install", "huggingface-hub==0.30.2"], check=True, env=env)
output = repo / "data/processed" / ("pa3-" + KIND + "-prep")
try:
    subprocess.run([sys.executable, "-m", "safeshift.pa3.native.prep", KIND,
        "--output", "data/processed/pa3-" + KIND + "-prep", "--execute-online-owner-prep"], cwd=repo, env=env, check=True)
finally:
    if output.is_dir():
        (output / "prep_source_input.json").write_text(json.dumps({
            "prep_source_commit": actual, "prep_bundle_sha256": sha256(bundle),
            "prep_release_sha256": sha256(receipt),
            "offline_source_commit": OFFLINE_SOURCE_COMMIT, "execution_authority": False,
        }, indent=2) + "\n")
# Exactly these files (paths relative to the printed folder); nothing else.
download = {"runtime": ["runtime-archive/pa3_runtime.tar.gz", "pa3_runtime_inventory.json",
                        "runtime_prep.json", "resolved-requirements.txt", "constraints.txt",
                        "commands.json", "prep_source_input.json"],
            "model": ["pa3_model.tar.gz", "model_prep.json", "prep_source_input.json"]}[KIND]
prep_receipt = json.loads((output / (KIND + "_prep.json")).read_text())
if KIND == "runtime":
    print("STATUS:", prep_receipt["status"])
    print("CHECKS:", json.dumps(prep_receipt["checks"], indent=1))
    print("INVENTORY_SHA256 (record separately):", prep_receipt["inventory_sha256"])
print("DOWNLOAD FROM:", output)
for name in download:
    print(sha256(output / name), (output / name).stat().st_size, name)
